In [ ]:
# Cell 1: Setup and Imports
import sys
import os
import logging
from pathlib import Path
from python_pypowsybl.Code.core.visualizer import NetworkVisualizer
from python_pypowsybl.Code.core.mo_topology import MoTopologyToolkit
from python_pypowsybl.Code.core.powerflow import PowerFlowRunner
from python_pypowsybl.Code.core.comparison import NetworkParameterComparator
from IPython.display import SVG, display, HTML

# Configure notebook logging to show INFO messages
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

# Add Code directory to path
current_dir = os.getcwd()
scripts_path = os.path.join(current_dir, "../Code")
if scripts_path not in sys.path:
    sys.path.append(scripts_path)


# Configurations
SOURCE_DIR = "../Models"
MODEL_NAME = "Dynawo.Examples.BESS.WECC.MyBESS_static"
DYNAWO_PKG = "/home/guiu/Projects/Dynawo/nightly/dynawo/ddb/Dynawo/package.mo"
LOCAL_FILES = ["MyBESS_static.mo", "BESS_init.mo"]
JSON_MODELS_PATH = "Models/parsed_models_data.json"
STR_NAME = "BESS"
EXPORT_FOLDER = f"grid_export_{STR_NAME}"

# Create the centralized export directory
os.makedirs(EXPORT_FOLDER, exist_ok=True)

print("Toolkit loaded.")

In [ ]:
# Instantiate the Toolkit (Facade)
toolkit = MoTopologyToolkit(SOURCE_DIR, MODEL_NAME, DYNAWO_PKG, LOCAL_FILES)

# Execute Parsing Pipeline
parsed_data = toolkit.parse_electrical_data()

# Export for verification into the centralized folder
json_filepath = os.path.join(EXPORT_FOLDER, f"grid_export_{STR_NAME}.json")
toolkit.export_to_standard_json(parsed_data, json_filepath)

# Import for verification
parsed_data = toolkit.import_from_standard_json(json_filepath)

print(f"Parsing successful. Found {len(parsed_data['generators'])} generators.")

In [ ]:
# Cell 3: Build PowSybl Network

parsed_data["source_xiidm"] = Path("om_powsybl_data/Base_Case.xiidm").resolve()
network = toolkit.build_powsybl_network(parsed_data)

print("Network constructed successfully.")
print(f"- Buses: {len(network.get_buses())}")
print(f"- Lines: {len(network.get_lines())}")
print(f"- Generators: {len(network.get_generators())}")
print(f"- Loads: {len(network.get_loads())}")
print(f"- Shunts: {len(network.get_shunt_compensators())}")
print(f"- Transformers: {len(network.get_2_windings_transformers())}")

In [ ]:
# Cell 4: Network Visualization

print("Rendering Network Diagrams...")
diagrams = NetworkVisualizer.generate_full_system_diagrams(network)

# 1. Display Macro View
if "network_area" in diagrams:
    display(HTML("<h3>Macro View: Network Area (Lines & Substations)</h3>"))
    display(SVG(str(diagrams["network_area"])))

# 2. Display Micro Views (Generators, Loads, etc.)
display(HTML("<h3>Micro View: Substations Details (Generators visible)</h3>"))
for name, svg in diagrams.items():
    if name != "network_area":
        display(HTML(f"<b>Substation: {name}</b>"))
        display(SVG(str(svg)))

In [ ]:
# Cell 5: AC Load Flow Execution and Results
from python_pypowsybl.Code.core.powerflow import PowerFlowRunner
from IPython.display import display

print("Starting AC Load Flow analysis...")

# 1. Execute the Load Flow (logs will be saved to EXPORT_FOLDER if it diverges)
is_converged = PowerFlowRunner.run_ac_loadflow(network, export_path=EXPORT_FOLDER)

if is_converged:
    print("\nSUCCESS: Load flow CONVERGED!")

    # 2. Extract Bus Data (Voltages and Angles)
    print("\n--- BUS RESULTS (Voltages & Angles) ---")
    try:
        buses_df = network.get_buses()
        # In newer versions, we focus on voltage magnitude and angle
        # P and Q injections are viewed at the equipment level or via 'v_mag'/'v_angle'
        columns_buses = ["v_mag", "v_angle"]
        display(buses_df[columns_buses])
    except Exception as e:
        print(f"Could not retrieve Bus results: {e}")

    # 3. Extract Generator Data (To see P and Q output)
    print("\n--- GENERATOR RESULTS (Active & Reactive Power) ---")
    try:
        gens_df = network.get_generators()
        # 'p' and 'q' are the calculated values after loadflow
        columns_gens = ["bus_id", "p", "q", "target_p", "target_v"]
        display(gens_df[columns_gens])
    except Exception as e:
        print(f"Could not retrieve Generator results: {e}")

    # 4. Extract Line Data
    print("\n--- LINE RESULTS (Power Flows & Currents) ---")
    try:
        lines_df = network.get_lines()
        columns_lines = ["bus1_id", "bus2_id", "p1", "q1", "p2", "q2", "i1", "i2"]
        display(lines_df[columns_lines])
    except Exception as e:
        print(f"Could not retrieve Line results: {e}")
else:
    print("\nFAILED: Load flow did NOT converge.")

# Save network to the centralized folder
toolkit.save_powsybl_network(network, export_path=EXPORT_FOLDER)

In [ ]:
# Cell 6: Cross-Validation with OpenModelica (OMPython)
from python_pypowsybl.Code.core.comparison import LoadFlowComparator

print("Running Ground Truth Validation...")
print("1. Compiling and simulating the Modelica model via OMC...")

# This will trigger the OMC compiler to solve the non-linear system (t=0.0)
# It might take ~10-30 seconds depending on the Nordic 32 complexity.
comparison_df = LoadFlowComparator.compare_voltages(
    network=network,
    connector=toolkit.connector,
    model_name=MODEL_NAME,
    parsed_data=parsed_data,
)

if not comparison_df.empty:
    print("\nSUCCESS: Validation completed! Showing top discrepancies (sorted by Δ V):")

    # Highlight cells with an error larger than 0.001 pu (0.1%)
    def highlight_errors(val):
        color = "orange" if isinstance(val, (int, float)) and val > 0.001 else "black"
        return f"color: {color}"

    # Apply style to error columns (CAMBIADO a .map)
    styled_df = comparison_df.style.map(
        highlight_errors, subset=["Δ V (pu)", "Δ Theta (deg)"]
    ).format("{:.4f}", na_rep="N/A")

    display(styled_df)

    # Print summary metrics
    mean_v_err = comparison_df["Δ V (pu)"].mean()
    mean_th_err = comparison_df["Δ Theta (deg)"].mean()
    print(f"\n--- Benchmark Summary ---")
    print(f"Average Voltage Mismatch: {mean_v_err:.6f} pu")
    print(f"Average Angle Mismatch:   {mean_th_err:.6f} deg")
else:
    print("Validation failed. Could not retrieve comparison data.")

In [ ]:
NetworkParameterComparator.generate_comparison_csv(
    om_data=parsed_data,
    json_path=os.path.join(EXPORT_FOLDER, f"grid_export_{STR_NAME}.json"),
    xiidm_path=os.path.join(EXPORT_FOLDER, f"grid_export_{STR_NAME}.xiidm"),
    export_path=EXPORT_FOLDER,
    output_prefix=f"compare_parameters_{STR_NAME}_",
)

In [ ]:
# Cell 7: Dynamic Model Linking and Audit
from python_pypowsybl.Code.core.model_linker import link_models

print("Resolving Modelica equations to IIDM static components...\n")

mapping, linked_registry = link_models(network)

if mapping is not None:
    print("=== DYNAMICALLY LINKED MODELS REGISTRY ===")
    for category, df in linked_registry.items():
        print(f"\n--- {category.upper()} ---")
        # Display only the relevant routing columns for clarity
        print(df.to_string(index=False))
else:
    print("CRITICAL ERROR: Model linking failed.")

# Comentario: Esto está bien por el momento, pero lo ideal es usar los modelos modelica y sus correspondientes INIT para compilarlos y que sean exacatemnte los modelos modelica que tocan

In [ ]:
import os
import re
import datetime

config_dir = os.path.expanduser("~/.itools")
config_file = os.path.join(config_dir, "config.yml")
dynawo_sys_path = None

# 1. Check if config.yml exists and read its content
if os.path.exists(config_file):
    with open(config_file, "r") as f:
        content = f.read()
        # Search for the homeDir value under the dynawo section using regex
        match = re.search(r'homeDir:\s*["\']?([^"\'\n]+)["\']?', content)
        if match:
            dynawo_sys_path = match.group(1)

# 2. Verify if a path was extracted and if it actually exists on your computer
if not dynawo_sys_path or not os.path.exists(dynawo_sys_path):
    raise FileNotFoundError(
        f"Error! No valid Dynawo installation found on the system. "
        f"Detected path: '{dynawo_sys_path}'. Make sure Dynawo is installed and properly configured in {config_file}."
    )

print(f"Configuration correct! Dynawo installation validated at: {dynawo_sys_path}")

# Save the verified path in an environment variable to use in cell 2
os.environ["DYNAWO_SYS_PATH"] = dynawo_sys_path


config_dir = os.path.expanduser("~/.itools")
target_config = os.path.join(config_dir, "config.yml")
template_config = "./om_powsybl_data/config.yml"
current_dir = os.getcwd()
dynawo_sys_path = os.environ.get("DYNAWO_SYS_PATH")

# Ensure cell 1 was executed successfully
if not dynawo_sys_path:
    raise ValueError("The Dynawo path is not defined. Please run the previous cell first.")

if os.path.exists(target_config):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = os.path.join(config_dir, f"config_{timestamp}.yml")
    os.rename(target_config, backup_path)
    print(f"Backup created at: {backup_path}")

with open(template_config, "r") as f:
    template_content = f.read()

template_content = template_content.replace('"WORKING_DIR/dynawo"', f'"{dynawo_sys_path}"')
template_content = template_content.replace("WORKING_DIR", current_dir)

os.makedirs(config_dir, exist_ok=True)
with open(target_config, "w") as f:
    f.write(template_content)

print(f"\nDone! Template successfully configured and saved to {target_config}:\n")
print(template_content)

In [ ]:
# Cell 9: Dynamic Physical Parameters Configuration via DDB Parsing
# LF Energy Summit 2026: Modelica to PyPowSyBl Toolkit

import os
import xml.etree.ElementTree as ET

print("Parsing Dynawo DDB to extract physical parameters for dynamic simulation...")

TARGET_DIR = "om_powsybl_data"
os.makedirs(TARGET_DIR, exist_ok=True)

# Sensible defaults for small-signal and transient stability.
# Essential to avoid algebraic loop divergence during initial solver steps.
sensible_defaults = {
    "generator_H": "3.2",
    "generator_RaPu": "0.0025",
    "generator_XdPu": "1.8",
    "generator_XqPu": "1.7",
    "generator_XpdPu": "0.3",
    "generator_XpqPu": "0.55",
    "generator_XppdPu": "0.25",
    "generator_XppqPu": "0.25",
    "generator_Tpd0": "8.0",
    "generator_Tpq0": "0.4",
    "generator_Tppd0": "0.03",
    "generator_Tppq0": "0.05",
    "generator_XlPu": "0.15",
    "generator_SNom": "1000.0",
    "generator_UNom": "400.0",
    "generator_UNomHV": "400.0",
    "generator_UNomLV": "20.0",
}


def get_ddb_parameters(model_name: str) -> str:
    """Dynamically parses the model's desc.xml file to extract writable parameters."""
    ddb_path = f"dynawo/ddb/{model_name}.desc.xml"
    xml_lines = []

    if os.path.exists(ddb_path):
        tree = ET.parse(ddb_path)
        ns = {"dyn": "http://www.rte-france.com/dynawo"}

        for param in tree.getroot().findall(".//dyn:parameter", ns):
            if param.get("readOnly") == "false":
                p_name = param.get("name")
                p_type = param.get("valueType")

                if p_name in sensible_defaults:
                    p_val = sensible_defaults[p_name]
                else:
                    fallback = (
                        "0.0" if p_type == "DOUBLE" else ("0" if p_type == "INT" else "false")
                    )
                    p_val = param.get("defaultValue", fallback)

                xml_lines.append(f'        <par type="{p_type}" name="{p_name}" value="{p_val}"/>')
    else:
        print(f"CRITICAL WARNING: {ddb_path} not found.")

    return "\n".join(xml_lines)


# Generate dynamic parameter sets by iterating through all generators in the network to define parameters for every instance[cite: 21].
generators_xml = ""
for gen_id in network.get_generators().index:
    # Resolve the model name dynamically (falling back to the required example)
    model_name = "GeneratorSynchronousFourWindings"

    generators_xml += f'''
    <set id="{gen_id}">
{get_ddb_parameters(model_name)}
        
        <!-- PyPowSyBl IIDM References -->
        <reference type="DOUBLE" name="generator_P0Pu" origData="IIDM" origName="p_pu"/>
        <reference type="DOUBLE" name="generator_Q0Pu" origData="IIDM" origName="q_pu"/>
        <reference type="DOUBLE" name="generator_U0Pu" origData="IIDM" origName="v_pu"/>
        <reference type="DOUBLE" name="generator_UPhase0" origData="IIDM" origName="angle"/>
    </set>'''

# 1. Base Case Parameters: Inject extracted physics and IIDM steady-state references
basecase_content = f"""<?xml version="1.0" encoding="UTF-8"?>
<parametersSet xmlns="http://www.rte-france.com/dynawo">
{generators_xml}

    <set id="BESS_1">
        <par type="DOUBLE" name="bess_SNom" value="6.0"/>
        <reference type="DOUBLE" name="bess_P0Pu" origData="IIDM" origName="p_pu"/>
        <reference type="DOUBLE" name="bess_Q0Pu" origData="IIDM" origName="q_pu"/>
        <reference type="DOUBLE" name="bess_U0Pu" origData="IIDM" origName="v_pu"/>
        <reference type="DOUBLE" name="bess_UPhase0" origData="IIDM" origName="angle"/>
    </set>
</parametersSet>
"""

# 2. Network Parameters: Global macroscopic simulation constants
network_content = """<?xml version="1.0" encoding="UTF-8"?>
<parametersSet xmlns="http://www.rte-france.com/dynawo">
    <set id="Network">
        <par type="DOUBLE" name="line_currentLimit_maxTimeOperation" value="999.0"/>
        <par type="DOUBLE" name="load_alpha" value="1.0"/>
        <par type="DOUBLE" name="load_beta" value="2.0"/>
        <par type="DOUBLE" name="transformer_tolV" value="0.01"/>
        <par type="BOOL" name="VirtualBus_2_hasShortCircuitCapabilities" value="true"/>
    </set>
</parametersSet>
"""

# Overwrite the empty configuration files in the om_powsybl_data directory
with open(os.path.join(TARGET_DIR, "Base_Case.par"), "w", encoding="utf-8") as f:
    f.write(basecase_content)

with open(os.path.join(TARGET_DIR, "Network.par"), "w", encoding="utf-8") as f:
    f.write(network_content)

print(f"Successfully generated dynamic physical parameters in '{TARGET_DIR}'.")

In [ ]:
# Cell 10: Dynamic Events and Output Observers
import pypowsybl.dynamic as dyn

print("Configuring topological events and observers...")

# 1. Define Topological Event: 3-Phase Short Circuit on BESS terminal
events = dyn.EventMapping()
# Fault at t=1.0s, cleared after 150ms (duration=0.15s)
events.add_node_fault(
    static_id="VirtualBus_2", start_time=1.0, fault_time=0.15, r_pu=0.0, x_pu=0.0001
)

# 2. Configure Output Observers (Voltages & Angles)
outputs = dyn.OutputVariableMapping()
outputs.add_standard_model_curves("VirtualBus_1", "U_value")
outputs.add_standard_model_curves("VirtualBus_2", "U_value")

# 3. Setup Simulation Parameters
sim_parameters = dyn.Parameters(
    start_time=0.0,
    stop_time=1.0,
)

print("Events and Observers successfully mapped.")

In [ ]:
# Cell 11: Execute Dynawo Time-Domain Simulation
import matplotlib.pyplot as plt

print("Running dynamic simulation via Dynawo engine...")
simulation = dyn.Simulation()

results = simulation.run(
    network,
    model_mapping=mapping,
    event_mapping=events,
    timeseries_mapping=outputs,
    parameters=sim_parameters,
)

if results.status().name == "SUCCESS":
    print("SUCCESS: Time-domain simulation completed.")

    # Extract and Plot State Trajectories
    curves = results.curves()
    plt.figure(figsize=(10, 5))

    # Plot Voltage Responses
    plt.plot(
        curves.index,
        curves["NETWORK_VirtualBus_2_U_value"],
        label="BESS Bus (Faulted)",
        color="red",
    )
    plt.plot(
        curves.index,
        curves["NETWORK_VirtualBus_1_U_value"],
        label="Slack Bus",
        linestyle="--",
        color="blue",
    )

    plt.title("Transient Response: 3-Phase Short Circuit at BESS Terminal")
    plt.xlabel("Time (s)")
    plt.ylabel("Voltage (PU)")
    plt.grid(True)
    plt.legend()
    plt.show()
else:
    print(f"CRITICAL ERROR: Simulation failed. Status: {results.status_text()}")